# GPT-style

Radford et al. 2018/2019 (GPT/GPT-2); rotary variant: Su et al. 2021, RoFormer ([arXiv:2104.09864](https://arxiv.org/abs/2104.09864)).

Decoder-only, causal self-attention, pre-norm blocks, next-character prediction on real Tiny Shakespeare. `pos_encoding` is a flag: `"learned"` (GPT-2's real choice) or `"rope"` (rotary, the modern default).

In [ ]:
import sys
sys.path.insert(0, '../..')
sys.path.insert(0, '.')

import torch
import torch.nn as nn
import matplotlib.pyplot as plt

from transformer_playground.data import load_tiny_shakespeare
from transformer_playground.device import resolve_device
from model import GPTModel

device = resolve_device('auto')
print('device:', device)

In [ ]:
text = load_tiny_shakespeare()
chars = sorted(set(text))
stoi = {c: i for i, c in enumerate(chars)}
data = torch.tensor([stoi[c] for c in text], dtype=torch.long)
n_val = len(data) // 10
train_data, val_data = data[:-n_val], data[-n_val:]
block_size, batch_size = 64, 32

def make_batches(data, block_size, batch_size):
    n = data.shape[0] - block_size - 1
    idx = torch.randint(0, n, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in idx]).to(device)
    y = torch.stack([data[i+1:i+block_size+1] for i in idx]).to(device)
    return x, y

print(f'{len(train_data)} train chars, {len(val_data)} val chars, vocab {len(chars)}')

In [ ]:
pos_encoding = 'rope'  # or 'learned'
model = GPTModel(vocab_size=len(chars), d_model=128, n_heads=4, n_layers=4, d_ff=512, max_len=block_size, pos_encoding=pos_encoding).to(device)
opt = torch.optim.Adam(model.parameters(), lr=3e-4)
loss_fn = nn.CrossEntropyLoss()

history = {'train_loss': []}
for step in range(300):
    x, y = make_batches(train_data, block_size, batch_size)
    logits = model(x)
    loss = loss_fn(logits.reshape(-1, logits.shape[-1]), y.reshape(-1))
    opt.zero_grad(); loss.backward(); opt.step()
    history['train_loss'].append(loss.item())
    if step % 50 == 0:
        print(f'step {step:4d} | train_loss {loss.item():.4f}')

In [ ]:
plt.plot(history['train_loss'])
plt.xlabel('step')
plt.ylabel('train loss')
plt.title(f'GPT-style ({pos_encoding}) training loss, real Tiny Shakespeare')
plt.show()